# Working with VLM Run in Pixeltable

Pixeltable's VLM Run integration lets you use VLM Run's multimodal AI capabilities directly from tables—chat completions, image generation, image editing, annotation, and video generation—without leaving your data workflows.

## What is VLM Run?

VLM Run is a multimodal AI platform with an OpenAI-compatible chat completions API and specialized toolsets for:

- **Understanding**: Analyze images, documents, and video with natural language
- **Image generation**: Create new images from text prompts
- **Image editing**: Edit existing images with natural language instructions
- **Visualization**: Annotate images with bounding boxes, keypoints, and segmentation masks
- **Video generation**: Create videos from text prompts

### Documentation

- [VLM Run Documentation](https://docs.vlm.run/)
- [VLM Run Dashboard](https://app.vlm.run/dashboard)

## Prerequisites

- A VLM Run account with an API key (see <https://app.vlm.run/dashboard>)

**Important:** VLM Run API calls consume credits based on your plan—monitor your usage to avoid unexpected charges.

In [ ]:
%pip install -qU pixeltable vlmrun openai

In [ ]:
import getpass
import os

if 'VLMRUN_API_KEY' not in os.environ:
    os.environ['VLMRUN_API_KEY'] = getpass.getpass('VLM Run API Key: ')

To read more about working with API keys in Pixeltable, see [Configuration](https://docs.pixeltable.com/platform/configuration).

## Setup

In [ ]:
import pixeltable as pxt
from pixeltable.functions import vlmrun

pxt.create_dir('vlmrun_demo', if_exists='replace_force')

## Chat Completions

The `vlmrun.chat_completions` UDF provides direct access to VLM Run's chat completions API. It accepts an OpenAI-compatible `messages` list and returns the raw API response as a dictionary.

### Text-only messages

In [ ]:
t = pxt.create_table('vlmrun_demo.text_chat', {'prompt': pxt.String}, if_exists='replace_force')

messages = [{'role': 'user', 'content': t.prompt}]
t.add_computed_column(response=vlmrun.chat_completions(messages))
t.add_computed_column(answer=t.response['choices'][0]['message']['content'])

t.insert(prompt='What is structured data extraction?')
t.select(t.prompt, t.answer).collect()

### Image understanding

Reference image columns directly in message content. Referenced media is uploaded to VLM Run automatically before each API call.

In [ ]:
t = pxt.create_table('vlmrun_demo.image_chat', {'image': pxt.Image}, if_exists='replace_force')

messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': 'Describe this image briefly'},
        {'type': 'image_url', 'image_url': t.image},
    ],
}]
t.add_computed_column(response=vlmrun.chat_completions(messages))
t.add_computed_column(description=t.response['choices'][0]['message']['content'])

t.insert(image='https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images/000000000036.jpg')
t.select(t.image, t.description).collect()

### Document understanding

Process PDFs and other documents by referencing a document column via `file_url`, combined with the `document` toolset.

In [ ]:
SAMPLE_PDF = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/tests/data/documents/layout-parser-paper.pdf'

t = pxt.create_table('vlmrun_demo.doc_chat', {'document': pxt.Document}, if_exists='replace_force')

messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': 'What is this document about? Summarize in one sentence.'},
        {'type': 'file_url', 'file_url': t.document},
    ],
}]
t.add_computed_column(
    response=vlmrun.chat_completions(
        messages, model_kwargs={'extra_body': {'toolsets': ['document']}}
    )
)
t.add_computed_column(summary=t.response['choices'][0]['message']['content'])

t.insert(document=SAMPLE_PDF)
t.select(t.summary).collect()

### Models

VLM Run offers three Orion 2 tiers:

| Model | Description |
|-------|-------------|
| `vlmrun-orion-2:fast` | Speed-optimized |
| `vlmrun-orion-2:auto` | Auto-select (default) |
| `vlmrun-orion-2:pro` | Most capable |

Orion 2 also lets you pin a specific backbone model (open-source options like Qwen 3.6 and Gemma 4, or proprietary options like GPT-5.5, Claude Opus 4.8, and Kimi 2.6) via the `model` parameter. See the [VLM Run documentation](https://docs.vlm.run/) for the full list of model variants.

## Image Generation

Use `vlmrun.generate_image` to create images from text prompts. It returns a `PIL.Image.Image` that Pixeltable stores as a native `pxt.Image` column.

In [ ]:
t = pxt.create_table('vlmrun_demo.image_gen', {'prompt': pxt.String}, if_exists='replace_force')
t.add_computed_column(image=vlmrun.generate_image(t.prompt))

t.insert(prompt='A futuristic cityscape at sunset with flying cars')
t.select(t.prompt, t.image).collect()

## Image Editing

Pass an image column via the `image` parameter to `generate_image` to edit an existing image according to your prompt.

In [ ]:
EXAMPLE_IMAGE = 'https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.agent/lunch-skyscraper.jpg'

t = pxt.create_table('vlmrun_demo.image_edit', {'image': pxt.Image}, if_exists='replace_force')
t.add_computed_column(
    blurred=vlmrun.generate_image('Blur all the faces in this image', image=t.image)
)

t.insert(image=EXAMPLE_IMAGE)
t.select(t.image, t.blurred).collect()

## Image Annotation (Visualization)

Use `vlmrun.annotate_image` to overlay bounding boxes, keypoints, or segmentation masks on an image using VLM Run's `viz` toolset. Unlike `generate_image`, the input image is required—you always need an image to annotate.

In [ ]:
EXAMPLE_IMAGE = 'https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.agent/lunch-skyscraper.jpg'

t = pxt.create_table('vlmrun_demo.annotation', {'image': pxt.Image}, if_exists='replace_force')
t.add_computed_column(
    annotated=vlmrun.annotate_image('Draw bounding boxes around all people', t.image)
)

t.insert(image=EXAMPLE_IMAGE)
t.select(t.image, t.annotated).collect()

## Video Generation

Use `vlmrun.generate_video` to create videos from text prompts. It returns a `pxt.Video` stored natively in Pixeltable.

**Note:** Video generation is asynchronous and may take a few minutes to complete.

In [ ]:
t = pxt.create_table('vlmrun_demo.video_gen', {'prompt': pxt.String}, if_exists='replace_force')
t.add_computed_column(video=vlmrun.generate_video(t.prompt))

t.insert(prompt='Create a 3 second long video of a red dot bouncing across the screen')
t.select(t.prompt, t.video).collect()

## Document Transformation (Redaction)

Use `vlmrun.generate_document` to transform a document and get the edited PDF back as a `pxt.Document` — for example, redacting PII at scale. Being explicit that you want the rendered PDF (not a text summary) produces the best results.

**Note:** very long documents can exceed the model's input-token limit; for multi-page PDFs, consider splitting into pages first.

In [ ]:
SAMPLE_PDF = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/tests/data/documents/layout-parser-paper.pdf'

t = pxt.create_table('vlmrun_demo.redaction', {'document': pxt.Document}, if_exists='replace_force')
t.add_computed_column(
    redacted=vlmrun.generate_document(
        'Produce a redacted version of this PDF: draw black redaction boxes over all author names and '
        'email addresses, keeping the rest visually identical. Return the rendered PDF file.',
        t.document,
    )
)

t.insert(document=SAMPLE_PDF)
t.select(t.document, t.redacted).collect()

## UDF Reference

| UDF | Input | Output | Toolset |
|-----|-------|--------|---------|
| `vlmrun.chat_completions` | `messages` list (media columns via `image_url` / `video_url` / `file_url`) | `dict` (raw API response) | Varies (via `model_kwargs`) |
| `vlmrun.generate_image` | `prompt`, optional `image` | `PIL.Image.Image` | `image-gen` |
| `vlmrun.annotate_image` | `prompt`, required `image` | `PIL.Image.Image` | `viz` |
| `vlmrun.generate_video` | `prompt`, optional `image` / `video` | `pxt.Video` | `video` |
| `vlmrun.generate_document` | `prompt`, required `document` | `pxt.Document` | `document` |

All UDFs accept `model` (default `'vlmrun-orion-2:auto'`) and `model_kwargs` for additional API parameters.

## Learn more

- VLM Run documentation: <https://docs.vlm.run/>
- Pixeltable documentation: <https://docs.pixeltable.com/>

If you build something with VLM Run, let us know!